# Phase 5: Anomaly Detection with SageMaker Random Cut Forest

**Goal:** Compare a supervised approach (XGBoost) against an **unsupervised** one (Random Cut Forest).

**Why this matters:**  
In production fraud detection, you often can't label every transaction. Unsupervised anomaly detection can flag suspicious activity *without* needing labeled fraud examples — but at the cost of more false positives and no direct probability calibration.

**SageMaker Random Cut Forest (RCF):**  
- A tree-based anomaly detection algorithm  
- Assigns an **anomaly score** to each point — higher = more anomalous  
- Fully managed as a SageMaker built-in algorithm (no training script needed)

In [ ]:
import sys
sys.path.insert(0, '..')

import boto3
import sagemaker
from sagemaker import RandomCutForest
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

from src.preprocessing import load_data, scale_features, split_data

%matplotlib inline

In [ ]:
BUCKET = 'your-bucket-name'   # <-- change this
PREFIX = 'fraud-detection'

session = sagemaker.Session()
role = sagemaker.get_execution_role()

## 1. Prepare Data

RCF is unsupervised — it trains on features only, with no labels.

In [ ]:
df = load_data('../data/raw/creditcard.csv')
df = scale_features(df)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

print(f'Training RCF on {X_train.shape[0]} unlabeled samples')
print(f'Evaluating on test set: {X_test.shape[0]} samples, {y_test.sum()} fraud')

In [ ]:
# Upload training features (no label column) to S3
import io

train_features_key = f'{PREFIX}/rcf/train/train_features.csv'
csv_buf = io.StringIO()
pd.DataFrame(X_train).to_csv(csv_buf, index=False, header=False)
boto3.client('s3').put_object(Bucket=BUCKET, Key=train_features_key, Body=csv_buf.getvalue())
rcf_train_uri = f's3://{BUCKET}/{train_features_key}'
print(f'Uploaded: {rcf_train_uri}')

## 2. Train RCF

RCF parameters:
- `num_trees`: more trees = more stable scores (default 50 is usually fine)
- `num_samples_per_tree`: how many points each tree sees; affects sensitivity
- Higher `num_samples_per_tree` → better for dense, high-dimensional data

In [ ]:
rcf = RandomCutForest(
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    data_location=rcf_train_uri,
    output_path=f's3://{BUCKET}/{PREFIX}/rcf/output',
    num_samples_per_tree=512,
    num_trees=50,
    sagemaker_session=session
)

rcf.fit(rcf.record_set(X_train.astype(np.float32)))
print('RCF training complete.')

## 3. Score the Test Set

We deploy a temporary endpoint to get anomaly scores, then delete it immediately.

In [ ]:
rcf_predictor = rcf.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
)
rcf_predictor.serializer = CSVSerializer()
rcf_predictor.deserializer = JSONDeserializer()
print('Endpoint deployed.')

In [ ]:
# Score in batches to avoid payload limits
BATCH_SIZE = 1000
scores = []

for i in range(0, len(X_test), BATCH_SIZE):
    batch = X_test[i:i + BATCH_SIZE].astype(np.float32)
    response = rcf_predictor.predict(batch)
    batch_scores = [r['score'] for r in response['scores']]
    scores.extend(batch_scores)
    if (i // BATCH_SIZE) % 10 == 0:
        print(f'  Scored {min(i + BATCH_SIZE, len(X_test))}/{len(X_test)}')

rcf_scores = np.array(scores)
print(f'\nScore range: [{rcf_scores.min():.2f}, {rcf_scores.max():.2f}]')

In [ ]:
# Always delete the endpoint when done — it costs money while running!
rcf_predictor.delete_endpoint()
print('Endpoint deleted.')

## 4. Evaluate: RCF Anomaly Scores vs Fraud Labels

In [ ]:
auc_roc = roc_auc_score(y_test, rcf_scores)
auc_pr  = average_precision_score(y_test, rcf_scores)

print(f'RCF (unsupervised) AUC-ROC: {auc_roc:.4f}')
print(f'RCF (unsupervised) AUC-PR:  {auc_pr:.4f}')

In [ ]:
# Distribution of anomaly scores by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

y_test_arr = np.array(y_test)
axes[0].hist(rcf_scores[y_test_arr == 0], bins=100, color='steelblue', alpha=0.7, density=True, label='Legitimate')
axes[0].hist(rcf_scores[y_test_arr == 1], bins=50, color='crimson', alpha=0.7, density=True, label='Fraud')
axes[0].set_title('RCF Anomaly Score Distribution')
axes[0].set_xlabel('Anomaly Score')
axes[0].legend()

# PR curve
prec, rec, _ = precision_recall_curve(y_test, rcf_scores)
axes[1].plot(rec, prec, color='darkorange', label=f'RCF (AUC-PR={auc_pr:.3f})')
axes[1].axhline(y=y_test.mean(), color='gray', linestyle='--', label='Random baseline')
axes[1].set_title('Precision-Recall Curve — RCF')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/rcf_results.png', dpi=120)
plt.show()

## 5. Supervised vs Unsupervised: Key Tradeoffs

| Dimension | XGBoost (supervised) | RCF (unsupervised) |
|---|---|---|
| Needs labeled fraud data | Yes | No |
| AUC-PR | ~0.80–0.88 | ~0.30–0.55 |
| Calibrated probabilities | Yes | No (anomaly scores only) |
| Adapts to new fraud patterns | No (requires retraining) | Somewhat (novelty sensitive) |
| Interpretable scores | Via SHAP | Not directly |
| Best use case | When you have labeled history | Cold start / new fraud types |

**Practical takeaway**: Use unsupervised anomaly detection as a *first-pass filter* or for flagging entirely new patterns. Use supervised models when you have labeled data — the performance gap is large.

**Next step → `05_threshold_tuning.ipynb`**: Take the best supervised model and tune the decision threshold to match your business requirements.